In [ ]:
import os
import sys
import warnings
from fractions import Fraction
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.collections import LineCollection
from matplotlib.colors import Normalize
from matplotlib.lines import Line2D
from mpl_toolkits.mplot3d.art3d import Line3DCollection
from matplotlib.ticker import AutoLocator, MultipleLocator
import numpy as np
import pandas as pd
import scipy.spatial as scispa
from cma import CMAEvolutionStrategy
from scipy.special import psi

plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['text.usetex'] = False

plt.rcParams['mathtext.fontset'] = 'custom'
plt.rcParams['mathtext.rm'] = 'Times New Roman'
plt.rcParams['mathtext.it'] = 'Times New Roman:italic'
plt.rcParams['mathtext.bf'] = 'Times New Roman:bold'

# ============================================================================
# 1. 核心互信息与量纲计算函数 (保持不变)
# ============================================================================

def calc_MI(x, y, mi_method="kraskov", mi_param=6):
    if mi_method == "binning":
        def entropy(data, num_bins):
            if data.ndim == 1: data = data.reshape(-1, 1)
            hist, _ = np.histogramdd(data, bins=[num_bins] * data.shape[1])
            p = hist / np.sum(hist)
            return -np.sum(p[p > 0] * np.log(p[p > 0]))
        return entropy(x, mi_param) + entropy(y, mi_param) - entropy(np.hstack([x, y]), mi_param)

    if mi_method == "kraskov":
        n_samples = x.shape[0]
        joint = np.hstack([x, y])
        kdtree = scispa.KDTree(joint)
        distances, _ = kdtree.query(joint, mi_param + 1, p=np.inf)
        radius = np.maximum(distances[:, -1], 1e-10)
        nx = scispa.KDTree(x).query_ball_point(x, radius, p=np.inf, return_length=True)
        ny = scispa.KDTree(y).query_ball_point(y, radius, p=np.inf, return_length=True)
        return psi(mi_param) - (psi(nx) + psi(ny)).mean() + psi(n_samples)
    raise ValueError(f"Unknown mi_method: {mi_method}")

def calc_pi(c, basis_matrices):
    c = np.asarray(c, dtype=float).reshape(1, -1)
    return np.dot(c, basis_matrices).flatten()

def coef_to_pi_mat(coef, X_data):
    safe_X = np.clip(X_data, a_min=1e-9, a_max=None)
    coef_floats = [float(v) for v in coef]
    return np.prod(safe_X ** coef_floats, axis=1).reshape(-1, 1)

def evaluate_c_mi(c_vec, X, Y, basis_matrices, mi_method, mi_param):
    coef_phys = calc_pi(c_vec, basis_matrices)
    pi_in = coef_to_pi_mat(coef_phys, X)
    mi_value = calc_MI(pi_in, Y, mi_method, mi_param)
    return float(mi_value)

def format_formula(coefs, vars_list):
    numerator, denominator = [], []
    for i, val in enumerate(coefs):
        if isinstance(val, Fraction):
            val_float = float(val)
            val_str = f"({val.numerator}/{val.denominator})" if val.denominator != 1 else str(val.numerator)
            is_zero, is_positive, is_one = val_float == 0, val_float > 0, abs(val) == 1
        else:
            val = round(float(val), 4)
            val_str = str(val)
            is_zero, is_positive, is_one = abs(val) < 1e-4, val > 0, abs(val) == 1.0

        if is_zero: continue
        if is_positive:
            item = vars_list[i] if is_one else f"{vars_list[i]}^{val_str}"
            numerator.append(item)
        else:
            if isinstance(val, Fraction):
                abs_val_str = f"({abs(val.numerator)}/{val.denominator})" if val.denominator != 1 else str(abs(val.numerator))
            else:
                abs_val_str = str(abs(val))
            item = vars_list[i] if is_one else f"{vars_list[i]}^{abs_val_str}"
            denominator.append(item)
    numerator_str = "*".join(numerator) if numerator else "1"
    denominator_str = "*".join(denominator)
    return f"({numerator_str})/({denominator_str})" if denominator_str else numerator_str

# ============================================================================
# 2. 优化历史记录和可视化
# ============================================================================

def _select_candidate_indices(fitness, sample_count, elite_count, rng):
    fitness = np.asarray(fitness, dtype=float)
    population_size = len(fitness)
    if population_size == 0: return np.array([], dtype=int)
    sample_count = max(1, min(int(sample_count), population_size))
    elite_count = max(1, min(int(elite_count), sample_count))
    sorted_idx = np.argsort(fitness)
    elite_idx = sorted_idx[:elite_count]
    remaining_count = sample_count - elite_count
    if remaining_count <= 0: return np.unique(elite_idx)
    pool = np.setdiff1d(np.arange(population_size), elite_idx, assume_unique=False)
    random_idx = rng.choice(pool, size=min(remaining_count, len(pool)), replace=False)
    return np.unique(np.concatenate([elite_idx, random_idx]))

def _choose_two_significant_basis_coefficients(summary_df, basic_pi_names, basic_pi_formulas):
    num_basis = len(basic_pi_names)
    if num_basis < 2: raise ValueError("基本无量纲数少于 2 个，无法生成二维响应面。")
    mean_cols = [f"mean_c_{i}" for i in range(num_basis)]
    best_cols = [f"best_c_{i}" for i in range(num_basis)]
    values = summary_df[mean_cols].to_numpy(dtype=float)
    robust_range = np.nanpercentile(values, 95, axis=0) - np.nanpercentile(values, 5, axis=0)
    trajectory_std = np.nanstd(values, axis=0)
    final_best = np.abs(summary_df[best_cols].iloc[-1].to_numpy(dtype=float))
    movement_score = robust_range + 0.25 * trajectory_std + 1e-6 * final_best
    order = np.argsort(movement_score)[::-1]
    pair = (int(order[0]), int(order[1]))
    diagnostics = pd.DataFrame({
        "basis_index": np.arange(num_basis, dtype=int),
        "coefficient": [f"c_{i + 1}" for i in range(num_basis)],
        "basic_pi_name": basic_pi_names,
        "basic_pi_formula": basic_pi_formulas,
        "movement_score": movement_score,
    }).sort_values("movement_score", ascending=False)
    return pair, diagnostics

def _evaluate_conditional_response_surface(history, idx_x, idx_y, grid_size):
    grid_size = max(15, int(grid_size))
    lower = np.asarray(history["bounds_lower"], dtype=float)
    upper = np.asarray(history["bounds_upper"], dtype=float)
    fixed_c = np.asarray(history["best_params"], dtype=float).copy()
    x_values = np.linspace(lower[idx_x], upper[idx_x], grid_size)
    y_values = np.linspace(lower[idx_y], upper[idx_y], grid_size)
    grid_x, grid_y = np.meshgrid(x_values, y_values)
    grid_mi = np.full_like(grid_x, np.nan, dtype=float)
    total = grid_size * grid_size
    completed = 0
    print(f"开始计算条件互信息响应面: {grid_size} × {grid_size} = {total} 个点")
    for row in range(grid_size):
        for col in range(grid_size):
            c_vec = fixed_c.copy()
            c_vec[idx_x] = grid_x[row, col]
            c_vec[idx_y] = grid_y[row, col]
            try:
                val = evaluate_c_mi(c_vec, history["X"], history["Y"], history["basis_matrices"], history["mi_method"], history["mi_param"])
                if np.isfinite(val): grid_mi[row, col] = val
            except: pass
            completed += 1
    return grid_x, grid_y, grid_mi, fixed_c

def _evaluate_projected_z(history, traj_x, traj_y, idx_x, idx_y, fixed_c):
    traj_z = []
    for x, y in zip(traj_x, traj_y):
        c_vec = fixed_c.copy()
        c_vec[idx_x] = x
        c_vec[idx_y] = y
        try:
            val = evaluate_c_mi(c_vec, history["X"], history["Y"], history["basis_matrices"], history["mi_method"], history["mi_param"])
        except: val = np.nan
        traj_z.append(val)
    return np.asarray(traj_z, dtype=float)

def format_latex_label(c_idx, pi_name):
    pi_name_tex = pi_name.replace("Π", r"\Pi").replace("_", r"_{") + "}" if "_" in pi_name else pi_name
    return f"$c_{{{c_idx}}}$ (${pi_name_tex}$)"

def save_history_tables(history, output_prefix):
    output_dir = os.path.dirname(output_prefix)
    if output_dir: os.makedirs(output_dir, exist_ok=True)
    history["summary"].to_csv(f"{output_prefix}_generation_history.csv", index=False)
    pd.DataFrame({
        "coefficient": [f"c_{i + 1}" for i in range(len(history["basic_pi_names"]))],
        "formula": history["basic_pi_formulas"],
    }).to_csv(f"{output_prefix}_basic_pi_definitions.csv", index=False)

def plot_optimization_history(
    history, input_vars, output_prefix, plot_config,
    basis_indices=None, response_surface_grid=101, trajectory_max_points=200, **kwargs
):
    summary_df = history["summary"]
    basic_pi_names = history["basic_pi_names"]
    basic_pi_formulas = history["basic_pi_formulas"]
    num_basis = len(basic_pi_names)

    if num_basis < 2: raise ValueError("响应面至少需要两个基本无量纲数。")
    output_dir = os.path.dirname(output_prefix)
    if output_dir: os.makedirs(output_dir, exist_ok=True)

    if basis_indices is None:
        (idx_x, idx_y), diagnostics = _choose_two_significant_basis_coefficients(summary_df, basic_pi_names, basic_pi_formulas)
    else:
        idx_x, idx_y = map(int, basis_indices)
        _, diagnostics = _choose_two_significant_basis_coefficients(summary_df, basic_pi_names, basic_pi_formulas)

    label_x = format_latex_label(idx_x + 1, basic_pi_names[idx_x])
    label_y = format_latex_label(idx_y + 1, basic_pi_names[idx_y])

    grid_x, grid_y, grid_mi, fixed_c = _evaluate_conditional_response_surface(
        history, idx_x, idx_y, response_surface_grid
    )

    if len(summary_df) > trajectory_max_points:
        indices = np.linspace(0, len(summary_df) - 1, trajectory_max_points, dtype=int)
        trajectory_df = summary_df.iloc[np.unique(indices)].reset_index(drop=True)
    else:
        trajectory_df = summary_df.copy()

    traj_x = trajectory_df[f"mean_c_{idx_x}"].to_numpy()
    traj_y = trajectory_df[f"mean_c_{idx_y}"].to_numpy()
    traj_z = _evaluate_projected_z(history, traj_x, traj_y, idx_x, idx_y, fixed_c)
    
    valid = np.isfinite(traj_x) & np.isfinite(traj_y) & np.isfinite(traj_z)
    traj_x, traj_y, traj_z = traj_x[valid], traj_y[valid], traj_z[valid]

    # =========================================================================
    # === 开始绘图 (极致空间优化版) ===
    # =========================================================================
    plt.rcParams.update({"text.usetex": False, "mathtext.fontset": "stix"})
    
    # 调整整体画布，稍微加宽一点，高度保持
    fig = plt.figure(figsize=(3.5, 7.5)) 
    
    # 🔥 核心修复：使用 5 行 GridSpec 解决 TypeError，实现自定义非对称间距
    # 行 0 (gs[0]): Colorbar 
    # 行 1 (gs[1]): 空白占位 (Colorbar 和 2D图 之间的微小间距)
    # 行 2 (gs[2]): 2D 图 (基础高度 1)
    # 行 3 (gs[3]): 空白占位 (2D图 和 3D图 之间的防遮挡间距)
    # 行 4 (gs[4]): 3D 图 (1.3 放大高度占比，补偿 3D自带的白边)
    gs = fig.add_gridspec(5, 1, height_ratios=[0.04, 0.05, 1, 0.4, 1.3], hspace=0.0)
    line_color = '#E63946'

    # ---------- 顶部 Colorbar (放在 gs[0]) ----------
    cbar_ax = fig.add_subplot(gs[0])

    # ---------- 中间 2D 等高线 (放在 gs[2]) ----------
    ax1 = fig.add_subplot(gs[2])
    contour = ax1.contourf(grid_x, grid_y, grid_mi, levels=32, cmap="viridis", alpha=0.9)
    ax1.plot(traj_x, traj_y, color=line_color, linewidth=2.5, zorder=4)
    ax1.scatter(traj_x[0], traj_y[0], color='white', edgecolor='black', marker='o', s=50, zorder=5, label='Start')
    ax1.scatter(traj_x[-1], traj_y[-1], color='gold', edgecolor='black', marker='*', s=80, zorder=5, label='End')
    ax1.set_xlabel(label_x, fontsize=14)
    ax1.set_ylabel(label_y, fontsize=14)
    ax1.tick_params(axis='both', labelsize=11)
    ax1.legend(loc="upper right", fontsize=11) # 固定在右上角防止遮挡中间的轨迹

    # ---------- 底部 3D 曲面 (放在 gs[4]) ----------
    ax2 = fig.add_subplot(gs[4], projection="3d")
    surf = ax2.plot_surface(grid_x, grid_y, grid_mi, cmap="viridis", alpha=0.9, linewidth=0, antialiased=True)
    z_offset = (np.nanmax(grid_mi) - np.nanmin(grid_mi)) * 0.015 
    ax2.plot(traj_x, traj_y, traj_z + z_offset, color=line_color, linewidth=2.5, zorder=10)
    ax2.scatter(traj_x[0], traj_y[0], traj_z[0] + z_offset, color='white', edgecolor='black', marker='o', s=50, alpha=1)
    ax2.scatter(traj_x[-1], traj_y[-1], traj_z[-1] + z_offset, color='gold', edgecolor='black', marker='*', s=80, alpha=1)

    # 提取绘图配置
    elev = plot_config.get("elev", 50)
    azim = plot_config.get("azim", 50)
    tick_pad = plot_config.get("tick_pad", -2)      
    label_pad = plot_config.get("label_pad", -3)    

    ax2.tick_params(axis='x', pad=tick_pad, labelsize=10)
    ax2.tick_params(axis='y', pad=tick_pad, labelsize=10)
    ax2.tick_params(axis='z', pad=tick_pad, labelsize=10)

    ax2.set_xlabel(f"\n{label_x}", fontsize=14, labelpad=label_pad)
    ax2.set_ylabel(f"\n{label_y}", fontsize=14, labelpad=label_pad)
    ax2.view_init(elev=elev, azim=azim)
    
    # 消除 3D 图的无效白边
    try:
        # 强制放大 3D 图内部的显示比例，挤掉自带白边 (Matplotlib 3.6+ 支持)
        ax2.set_box_aspect(None, zoom=1.15) 
    except TypeError:
        # 如果你环境里的 matplotlib 版本较低，走这个兼容逻辑
        ax2.dist = 8.5 
    
    # ---------- 绑定 Colorbar 到最上面的轴 ----------
    cbar = fig.colorbar(contour, cax=cbar_ax, orientation='horizontal')
    # cbar.set_label("Mutual Information", fontsize=15, labelpad=8) 
    cbar_ax.xaxis.set_ticks_position('top')
    cbar_ax.xaxis.set_label_position('top')
    cbar_ax.tick_params(labelsize=11)
    cbar_ax.xaxis.set_major_locator(MultipleLocator(0.5))
    # 保存时使用 bbox_inches='tight' 自动裁剪掉多余的白边
    fig.savefig(f"{output_prefix}_optimization_landscape.png", dpi=300, bbox_inches='tight', pad_inches=0.1)
    plt.show()

    return {
        "selected_coefficients": (f"c_{idx_x + 1}", f"c_{idx_y + 1}"),
        "selected_basic_pi": (basic_pi_names[idx_x], basic_pi_names[idx_y]),
    }

# ============================================================================
# 3. 单次 CMA-ES 优化、有理化与历史记录
# ============================================================================

def run_optimization(X, Y, basis_matrices, input_vars, config: dict):
    basis_matrices_np = np.asarray(basis_matrices, dtype=float)
    num_basis = basis_matrices_np.shape[0]

    basic_pi_names = config.get("basic_pi_names", [f"Π_{i + 1}" for i in range(num_basis)])
    basic_pi_formulas = [format_formula(row, input_vars) for row in basis_matrices_np]

    mi_method = config.get("mi_method", "kraskov")
    mi_param = config.get("mi_param", 6)
    
    options = {
        "bounds": [[config.get("cma_lower_bound", -2.0)] * num_basis,
                   [config.get("cma_upper_bound", 2.0)] * num_basis],
        "maxiter": config.get("cma_maxiter", 50000),
        "popsize": config.get("cma_popsize", 150),
        "seed": 42,
        "verbose": -9,
        "tolfun": 1e-6,
    }

    es = CMAEvolutionStrategy(
        [config.get("cma_init_position", 0.1)] * num_basis,
        config.get("cma_init_stepsize", 0.5),
        options,
    )

    generation_rows = []
    
    def objective_fn(cma_para):
        try:
            mi_value = evaluate_c_mi(cma_para, X, Y, basis_matrices_np, mi_method, mi_param)
            return -float(mi_value) if np.isfinite(mi_value) else 1e6
        except Exception:
            return 1e6

    while not es.stop():
        solutions = es.ask()
        fitness = np.asarray([objective_fn(s) for s in solutions], dtype=float)
        es.tell(solutions, fitness.tolist())

        generation = int(es.countiter)
        
        row = {
            "generation": generation,
            "global_best_mi": -float(es.result.fbest),
            "sigma": float(es.sigma),
        }
        for i, value in enumerate(es.mean): row[f"mean_c_{i}"] = float(value)
        for i, value in enumerate(es.result.xbest): row[f"best_c_{i}"] = float(value)
        generation_rows.append(row)

    best_params = np.asarray(es.result.xbest, dtype=float)
    float_mi = -float(es.result.fbest)

    history = {
        "summary": pd.DataFrame(generation_rows),
        "basis_matrices": basis_matrices_np.copy(),
        "input_vars": list(input_vars),
        "basic_pi_names": basic_pi_names,
        "basic_pi_formulas": basic_pi_formulas,
        "X": X, "Y": Y, "mi_method": mi_method, "mi_param": mi_param,
        "bounds_lower": np.full(num_basis, config.get("cma_lower_bound", -2.0), dtype=float),
        "bounds_upper": np.full(num_basis, config.get("cma_upper_bound", 2.0), dtype=float),
        "best_params": best_params.copy(),
    }

    # 有理化处理 (精简展示)
    rat_max_denom = config.get("rational_max_denom", 5)
    fallback_decimals = config.get("fallback_decimals", 2)
    
    def rationalize_c_vec(vec, max_d, max_err):
        rat_v = []
        for w in vec:
            if abs(w) < 1e-5: rat_v.append(Fraction(0, 1))
            else:
                frac = Fraction(float(w)).limit_denominator(max_d)
                rat_v.append(frac if abs(w - float(frac)) <= max_err else float(w))
        return rat_v

    rat_c_vec = rationalize_c_vec(best_params, rat_max_denom, config.get("rational_max_error", 0.05))
    rat_mi = evaluate_c_mi([float(x) for x in rat_c_vec], X, Y, basis_matrices_np, mi_method, mi_param)
    
    strategy_used = "Rationalized" if abs(float_mi - rat_mi)/(abs(float_mi)+1e-9) <= config.get("rational_max_mi_drop", 0.05) else "Decimal Fallback"
    final_c_vec = rat_c_vec if strategy_used == "Rationalized" else [round(float(w), fallback_decimals) for w in best_params]

    return float_mi, "formula_float", rat_mi, "formula_rat", strategy_used, best_params.tolist(), final_c_vec, history


# ============================================================================
# 🔥 4. 算例配置字典 (整合核心逻辑)
# ============================================================================

CASES = {'rayben': {'csv_path': '../../data/dataset_rayben.csv', 'output_dir': 'generated/rayben', 'input_vars': ['h', 'lambda', 'delta_T', 'g', 'alpha', 'nu', 'kappa'], 'output_var': 'Nu', 'basis_matrices': [[0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0], [-1.5, 0.0, 0.0, -0.5, 0.0, 1.0, 0.0], [-1.5, 0.0, 0.0, -0.5, 0.0, 0.0, 1.0]], 'cma_config': {'mi_param': 6, 'cma_init_position': 0.3, 'basic_pi_names': ['Π_1', 'Π_2', 'Π_3'], 'rational_max_denom': 4}, 'plot_config': {'elev': 50, 'azim': 50, 'tick_pad': -3, 'label_pad': -4}}, 'presGrad': {'csv_path': '../../data/dataset_presGrad.csv', 'output_dir': 'generated/presGrad', 'input_vars': ['U_inf', 'nu', 'MKE', 'delta', 'delta*', 'theta'], 'output_var': 'U_tau/U_inf', 'basis_matrices': [[-1.0, -1.0, 1.0, 0.0, 0.0, 0.0], [1.0, -1.0, 0.0, 1.0, 0.0, 0.0], [1.0, -1.0, 0.0, 0.0, 1.0, 0.0], [1.0, -1.0, 0.0, 0.0, 0.0, 1.0]], 'cma_config': {'mi_param': 3, 'cma_init_position': 0.1, 'basic_pi_names': ['Π_1', 'Π_2', 'Π_3', 'Π_4'], 'rational_max_denom': 5}, 'plot_config': {'elev': 60, 'azim': -70, 'tick_pad': -3, 'label_pad': -4}}, 'keyhole': {'csv_path': '../../data/dataset_keyhole.csv', 'output_dir': 'generated/keyhole', 'input_vars': ['etaP', 'Vs', 'rho', 'cp', 'Tl-T0', 'r0', 'alpha'], 'output_var': 'e*', 'basis_matrices': [[0.0, -2.0, 0.0, 1.0, 1.0, 0.0, 0.0], [-0.5, 1.5, 0.5, 0.0, 0.0, 1.0, 0.0], [-0.5, 0.5, 0.5, 0.0, 0.0, 0.0, 1.0]], 'cma_config': {'mi_param': 4, 'cma_init_position': 0.1, 'basic_pi_names': ['Π_1', 'Π_2', 'Π_3'], 'rational_max_denom': 5}, 'plot_config': {'elev': 50, 'azim': 50, 'tick_pad': -1.5, 'label_pad': -2}}, 'blasius': {'csv_path': '../../data/dataset_blasius.csv', 'output_dir': 'generated/blasius', 'input_vars': ['U', 'nu', 'x', 'y'], 'output_var': 'u/U', 'basis_matrices': [[1.0, -1.0, 1.0, 0.0], [1.0, -1.0, 0.0, 1.0]], 'cma_config': {'mi_param': 5, 'cma_popsize': 200, 'cma_init_stepsize': 0.3, 'cma_init_position': 0.6, 'basic_pi_names': ['Π_1', 'Π_2'], 'rational_max_denom': 8}, 'plot_config': {'elev': 65, 'azim': -20, 'tick_pad': -2, 'label_pad': -3}}}


In [ ]:


# ============================================================================
# 5. 执行主程序
# ============================================================================
if __name__ == "__main__":
    
    # 🔥 在这里修改你要运行的算例！可选值: "rayben", "presGrad", "keyhole", "blasius"
    ACTIVE_CASE = "rayben" 
    
    # 获取算例配置
    case_data = CASES[ACTIVE_CASE]
    CSV_PATH = case_data["csv_path"]
    VIZ_OUTPUT_DIR = case_data["output_dir"]
    KSG_K = case_data["cma_config"]["mi_param"]
    
    if not os.path.exists(CSV_PATH):
        print(f"❌ 未找到数据集文件: {CSV_PATH}，请检查路径是否正确。")
        sys.exit()
    os.makedirs(VIZ_OUTPUT_DIR, exist_ok=True)

    # CMA-ES 默认参数 (如果算例字典中没有指定，则用默认的)
    default_cma_config = {
        "mi_method": "kraskov", "cma_popsize": 300, "cma_maxiter": 50000, 
        "cma_init_stepsize": 0.5, "cma_lower_bound": -1.0, "cma_upper_bound": 1.0,
        "rational_max_error": 0.05, "rational_max_mi_drop": 0.06, "fallback_decimals": 2
    }
    # 将默认配置和特定算例的配置合并
    final_config = {**default_cma_config, **case_data["cma_config"]}

    df = pd.read_csv(CSV_PATH)
    df_clean = df[case_data["input_vars"] + [case_data["output_var"]]].dropna()
    X_data = np.asarray(df_clean[case_data["input_vars"]], dtype=float)
    Y_data = np.asarray(df_clean[[case_data["output_var"]]], dtype=float)

    print("==================================================")
    print(f"🧪 开始优化算例: {ACTIVE_CASE.upper()}")
    print(f"数据样本量: {X_data.shape[0]} | Kraskov k: {KSG_K}")
    print("==================================================\n")

    try:
        float_mi, float_formula, rational_mi, rational_formula, strategy, raw_params, rational_params, history = run_optimization(
            X_data, Y_data, case_data["basis_matrices"], case_data["input_vars"], final_config
        )

        output_prefix = os.path.join(VIZ_OUTPUT_DIR, f"single_ksg_k_{KSG_K}")
        save_history_tables(history, output_prefix)

        plot_info = plot_optimization_history(
            history, case_data["input_vars"], output_prefix, 
            plot_config=case_data["plot_config"], # 传入绘图参数
            response_surface_grid=101  
        )

        print("\n========================= 单次优化结果 =========================")
        print(f"有理化 c: {[str(p) for p in rational_params]}")
        print(f"有理化互信息: {rational_mi:.6f}")
        print("==============================================================")

    except Exception as exc:
        print(f"运行出错: {exc}")
        raise

In [ ]:


# ============================================================================
# 5. 执行主程序
# ============================================================================
if __name__ == "__main__":
    
    # 🔥 在这里修改你要运行的算例！可选值: "rayben", "presGrad", "keyhole", "blasius"
    ACTIVE_CASE = "presGrad"
    
    # 获取算例配置
    case_data = CASES[ACTIVE_CASE]
    CSV_PATH = case_data["csv_path"]
    VIZ_OUTPUT_DIR = case_data["output_dir"]
    KSG_K = case_data["cma_config"]["mi_param"]
    
    if not os.path.exists(CSV_PATH):
        print(f"❌ 未找到数据集文件: {CSV_PATH}，请检查路径是否正确。")
        sys.exit()
    os.makedirs(VIZ_OUTPUT_DIR, exist_ok=True)

    # CMA-ES 默认参数 (如果算例字典中没有指定，则用默认的)
    default_cma_config = {
        "mi_method": "kraskov", "cma_popsize": 300, "cma_maxiter": 50000, 
        "cma_init_stepsize": 0.5, "cma_lower_bound": -1.0, "cma_upper_bound": 1.0,
        "rational_max_error": 0.05, "rational_max_mi_drop": 0.06, "fallback_decimals": 2
    }
    # 将默认配置和特定算例的配置合并
    final_config = {**default_cma_config, **case_data["cma_config"]}

    df = pd.read_csv(CSV_PATH)
    df_clean = df[case_data["input_vars"] + [case_data["output_var"]]].dropna()
    X_data = np.asarray(df_clean[case_data["input_vars"]], dtype=float)
    Y_data = np.asarray(df_clean[[case_data["output_var"]]], dtype=float)

    print("==================================================")
    print(f"🧪 开始优化算例: {ACTIVE_CASE.upper()}")
    print(f"数据样本量: {X_data.shape[0]} | Kraskov k: {KSG_K}")
    print("==================================================\n")

    try:
        float_mi, float_formula, rational_mi, rational_formula, strategy, raw_params, rational_params, history = run_optimization(
            X_data, Y_data, case_data["basis_matrices"], case_data["input_vars"], final_config
        )

        output_prefix = os.path.join(VIZ_OUTPUT_DIR, f"single_ksg_k_{KSG_K}")
        save_history_tables(history, output_prefix)

        plot_info = plot_optimization_history(
            history, case_data["input_vars"], output_prefix, 
            plot_config=case_data["plot_config"], # 传入绘图参数
            response_surface_grid=101  
        )

        print("\n========================= 单次优化结果 =========================")
        print(f"有理化 c: {[str(p) for p in rational_params]}")
        print(f"有理化互信息: {rational_mi:.6f}")
        print("==============================================================")

    except Exception as exc:
        print(f"运行出错: {exc}")
        raise

In [ ]:


# ============================================================================
# 5. 执行主程序
# ============================================================================
if __name__ == "__main__":
    
    # 🔥 在这里修改你要运行的算例！可选值: "rayben", "presGrad", "keyhole", "blasius"
    ACTIVE_CASE = "keyhole"
    
    # 获取算例配置
    case_data = CASES[ACTIVE_CASE]
    CSV_PATH = case_data["csv_path"]
    VIZ_OUTPUT_DIR = case_data["output_dir"]
    KSG_K = case_data["cma_config"]["mi_param"]
    
    if not os.path.exists(CSV_PATH):
        print(f"❌ 未找到数据集文件: {CSV_PATH}，请检查路径是否正确。")
        sys.exit()
    os.makedirs(VIZ_OUTPUT_DIR, exist_ok=True)

    # CMA-ES 默认参数 (如果算例字典中没有指定，则用默认的)
    default_cma_config = {
        "mi_method": "kraskov", "cma_popsize": 300, "cma_maxiter": 50000, 
        "cma_init_stepsize": 0.5, "cma_lower_bound": -1.0, "cma_upper_bound": 1.0,
        "rational_max_error": 0.05, "rational_max_mi_drop": 0.1, "fallback_decimals": 2
    }
    # 将默认配置和特定算例的配置合并
    final_config = {**default_cma_config, **case_data["cma_config"]}

    df = pd.read_csv(CSV_PATH)
    df_clean = df[case_data["input_vars"] + [case_data["output_var"]]].dropna()
    X_data = np.asarray(df_clean[case_data["input_vars"]], dtype=float)
    Y_data = np.asarray(df_clean[[case_data["output_var"]]], dtype=float)

    print("==================================================")
    print(f"🧪 开始优化算例: {ACTIVE_CASE.upper()}")
    print(f"数据样本量: {X_data.shape[0]} | Kraskov k: {KSG_K}")
    print("==================================================\n")

    try:
        float_mi, float_formula, rational_mi, rational_formula, strategy, raw_params, rational_params, history = run_optimization(
            X_data, Y_data, case_data["basis_matrices"], case_data["input_vars"], final_config
        )

        output_prefix = os.path.join(VIZ_OUTPUT_DIR, f"single_ksg_k_{KSG_K}")
        save_history_tables(history, output_prefix)

        plot_info = plot_optimization_history(
            history, case_data["input_vars"], output_prefix, 
            plot_config=case_data["plot_config"], # 传入绘图参数
            response_surface_grid=101  
        )

        print("\n========================= 单次优化结果 =========================")
        print(f"有理化 c: {[str(p) for p in rational_params]}")
        print(f"有理化互信息: {rational_mi:.6f}")
        print("==============================================================")

    except Exception as exc:
        print(f"运行出错: {exc}")
        raise

In [ ]:


# ============================================================================
# 5. 执行主程序
# ============================================================================
if __name__ == "__main__":
    
    # 🔥 在这里修改你要运行的算例！可选值: "rayben", "presGrad", "keyhole", "blasius"
    ACTIVE_CASE = "blasius"
    
    # 获取算例配置
    case_data = CASES[ACTIVE_CASE]
    CSV_PATH = case_data["csv_path"]
    VIZ_OUTPUT_DIR = case_data["output_dir"]
    KSG_K = case_data["cma_config"]["mi_param"]
    
    if not os.path.exists(CSV_PATH):
        print(f"❌ 未找到数据集文件: {CSV_PATH}，请检查路径是否正确。")
        sys.exit()
    os.makedirs(VIZ_OUTPUT_DIR, exist_ok=True)

    # CMA-ES 默认参数 (如果算例字典中没有指定，则用默认的)
    default_cma_config = {
        "mi_method": "kraskov", "cma_popsize": 300, "cma_maxiter": 50000, 
        "cma_init_stepsize": 0.5, "cma_lower_bound": -1.0, "cma_upper_bound": 1.0,
        "rational_max_error": 0.05, "rational_max_mi_drop": 0.06, "fallback_decimals": 2
    }
    # 将默认配置和特定算例的配置合并
    final_config = {**default_cma_config, **case_data["cma_config"]}

    df = pd.read_csv(CSV_PATH)
    df_clean = df[case_data["input_vars"] + [case_data["output_var"]]].dropna()
    X_data = np.asarray(df_clean[case_data["input_vars"]], dtype=float)
    Y_data = np.asarray(df_clean[[case_data["output_var"]]], dtype=float)

    print("==================================================")
    print(f"🧪 开始优化算例: {ACTIVE_CASE.upper()}")
    print(f"数据样本量: {X_data.shape[0]} | Kraskov k: {KSG_K}")
    print("==================================================\n")

    try:
        float_mi, float_formula, rational_mi, rational_formula, strategy, raw_params, rational_params, history = run_optimization(
            X_data, Y_data, case_data["basis_matrices"], case_data["input_vars"], final_config
        )

        output_prefix = os.path.join(VIZ_OUTPUT_DIR, f"single_ksg_k_{KSG_K}")
        save_history_tables(history, output_prefix)

        plot_info = plot_optimization_history(
            history, case_data["input_vars"], output_prefix, 
            plot_config=case_data["plot_config"], # 传入绘图参数
            response_surface_grid=101  
        )

        print("\n========================= 单次优化结果 =========================")
        print(f"有理化 c: {[str(p) for p in rational_params]}")
        print(f"有理化互信息: {rational_mi:.6f}")
        print("==============================================================")

    except Exception as exc:
        print(f"运行出错: {exc}")
        raise